In [1]:
import json
import sys
from pathlib import Path

project_root = Path(r"C:\Users\HP\Documents\Projects\Brain_Tumor_XAI")
protocol_path = project_root / "configs" / "brisc_training_protocol_draft_v1.json"

with protocol_path.open(encoding="utf-8-sig") as file:
    protocol = json.load(file)

print("Python executable:", sys.executable)
print("Loaded protocol:", protocol_path)
print("Protocol status:", protocol["status"])

print("\nPretrained candidates:")
for model_id in protocol["model_ids"]:
    print(" -", model_id)

augmentation = protocol["augmentation"]
print("\nProposed translation limit per axis:",
      f"{augmentation['proposed_translation_fraction_per_axis']:.0%}")
print("Translation review:", augmentation["translation_status"])

Python executable: C:\Users\HP\miniconda3\envs\brainxai-train\python.exe
Loaded protocol: C:\Users\HP\Documents\Projects\Brain_Tumor_XAI\configs\brisc_training_protocol_draft_v1.json
Protocol status: draft_for_review

Pretrained candidates:
 - mobilenetv4_conv_small.e2400_r224_in1k
 - convnextv2_atto.fcmae_ft_in1k
 - repvit_m0_9.dist_450e_in1k

Proposed translation limit per axis: 2%
Translation review: pending independent review


In [2]:
import platform

import timm
import torch
import torchvision

cuda_available = torch.cuda.is_available()
current_environment = {
    "python": platform.python_version(),
    "torch": str(torch.__version__),
    "torchvision": str(torchvision.__version__),
    "timm": timm.__version__,
    "gpu": torch.cuda.get_device_name(0) if cuda_available else "CUDA unavailable",
}

print("CUDA available:", cuda_available)

for key, actual in current_environment.items():
    expected = protocol["environment"][key]
    if actual == expected:
        print(f"{key}: {actual} [match]")
    else:
        print(f"{key}: {actual} [expected: {expected}]")

C:\Users\HP\miniconda3\envs\brainxai-train\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


CUDA available: True
python: 3.11.18 [match]
torch: 2.9.1+cu126 [match]
torchvision: 0.24.1+cu126 [match]
timm: 1.0.31 [match]
gpu: NVIDIA GeForce GTX 1650 [match]


In [3]:
import csv
from hashlib import sha256

processed_dir = project_root / "data" / "processed"
expected_hashes = protocol["development_manifest_sha256"]
records = {}

for split in ("train", "validation"):
    path = processed_dir / f"classification_{split}_manifest.csv"
    actual_hash = sha256(path.read_bytes()).hexdigest()

    if actual_hash != expected_hashes[split]:
        raise ValueError(
            f"{split} manifest differs from Notebook 08.\n"
            f"File: {path}\n"
            f"Expected: {expected_hashes[split]}\n"
            f"Found:    {actual_hash}"
        )

    with path.open(encoding="utf-8-sig", newline="") as file:
        records[split] = list(csv.DictReader(file))

    print(f"{split}: {len(records[split]):,} rows | SHA-256 match")

print("PASS | Both development manifests verified")

train: 3,889 rows | SHA-256 match
validation: 973 rows | SHA-256 match
PASS | Both development manifests verified


In [4]:
from copy import deepcopy

config_path = processed_dir / "brisc_preprocessing_config.json"
config_bytes = config_path.read_bytes()
expected_hash = protocol["augmentation"]["base_config_sha256"]

if sha256(config_bytes).hexdigest() != expected_hash:
    raise ValueError(f"Preprocessing config differs from Notebook 08: {config_path}")

preprocessing_config = json.loads(config_bytes)
base_policy = preprocessing_config["augmentation"]

assert base_policy["apply_to_train_only"] is True
assert base_policy["apply_to_validation"] is False
assert base_policy["apply_to_test"] is False

# Keep the frozen configuration on disk and use a copy for this run.
aug_policy = deepcopy(base_policy)
limit = protocol["augmentation"]["proposed_translation_fraction_per_axis"]
aug_policy["translation_fraction"].update(horizontal=limit, vertical=limit)

print("Preprocessing config: SHA-256 match")
for axis in ("horizontal", "vertical"):
    original = base_policy["translation_fraction"][axis]
    working = aug_policy["translation_fraction"][axis]
    print(f"{axis}: {original:.0%} in saved config -> {working:.0%} for timed run")

print("Augmentation applies to training only")
print("Translation review:", protocol["augmentation"]["translation_status"])

Preprocessing config: SHA-256 match
horizontal: 5% in saved config -> 2% for timed run
vertical: 5% in saved config -> 2% for timed run
Augmentation applies to training only
Translation review: pending independent review


In [5]:
from PIL import Image
from torchvision.transforms.functional import pil_to_tensor

dataset_root = project_root / "data" / "raw" / "brisc2025"
image_size = preprocessing_config["classification"]["default_input_size"]

def load_canonical_mri(path):
    with Image.open(path) as image:
        gray = image.convert("L")
        resized = gray.resize(
            (image_size, image_size),
            resample=Image.Resampling.BILINEAR,
        )
    single_channel = pil_to_tensor(resized).to(torch.float32) / 255.0
    return single_channel.repeat(3, 1, 1)

sample_row = records["train"][0]
sample_image = load_canonical_mri(dataset_root / sample_row["relative_path"])

print("Training scan:", sample_row["filename"])
print("Class:", sample_row["tumor_label"])
print("Tensor shape:", tuple(sample_image.shape))
print("Data type:", sample_image.dtype)
print("Pixel range:", sample_image.min().item(), "to", sample_image.max().item())

Training scan: brisc2025_train_01458_me_ax_t1.jpg
Class: meningioma
Tensor shape: (3, 224, 224)
Data type: torch.float32
Pixel range: 0.0 to 0.9764705896377563


In [6]:
from torchvision import transforms as T
from torchvision.transforms import InterpolationMode

translation = aug_policy["translation_fraction"]
train_augmentation = T.Compose([
    T.RandomAffine(
        degrees=tuple(aug_policy["rotation_degrees"]),
        translate=(translation["horizontal"], translation["vertical"]),
        scale=tuple(aug_policy["scale_range"]),
        interpolation=InterpolationMode.BILINEAR,
        fill=0,
    ),
    T.ColorJitter(
        brightness=tuple(aug_policy["brightness_factor_range"]),
        contrast=tuple(aug_policy["contrast_factor_range"]),
    ),
])

def augment_training_image(image, run_seed, epoch, row_index):
    # row_index is the scan's position in the frozen training manifest.
    key = f"{run_seed}:{epoch}:{row_index}".encode("utf-8")
    draw_seed = int.from_bytes(sha256(key).digest()[:8], byteorder="big")

    with torch.random.fork_rng(devices=[]):
        torch.default_generator.manual_seed(draw_seed)
        return train_augmentation(image)

# Cell 5 loaded row 0 of the training manifest.
rng_before = torch.get_rng_state().clone()
first = augment_training_image(sample_image, run_seed=42, epoch=1, row_index=0)
repeat = augment_training_image(sample_image, run_seed=42, epoch=1, row_index=0)

assert torch.equal(first, repeat), "Repeated augmentation produced different pixels."
assert torch.equal(torch.get_rng_state(), rng_before), "CPU random state changed."

print("Training scan:", sample_row["filename"])
print("Translation limits:", train_augmentation.transforms[0].translate)
print("PASS | Augmentation repeats exactly and preserves the CPU random state")

Training scan: brisc2025_train_01458_me_ax_t1.jpg
Translation limits: (0.02, 0.02)
PASS | Augmentation repeats exactly and preserves the CPU random state


In [7]:
def make_training_batches(n_images, batch_size, generator):
    order = torch.randperm(n_images, generator=generator).tolist()
    batches = [
        order[start:start + batch_size]
        for start in range(0, n_images, batch_size)
    ]

    if len(batches) >= 2 and len(batches[-1]) == 1:
        batches[-1].insert(0, batches[-2].pop())

    return batches


n_train = len(records["train"])
batch_size = protocol["batch_size"]
check_generator = torch.Generator().manual_seed(protocol["seeds"][0])
check_batches = make_training_batches(n_train, batch_size, check_generator)

indices = [index for batch in check_batches for index in batch]
final_sizes = [len(batch) for batch in check_batches[-2:]]
loader_settings = protocol["training_loader"]

assert sorted(indices) == list(range(n_train)), "Missing or repeated training row"
assert len(check_batches) == loader_settings["batches_per_epoch"]
assert final_sizes == loader_settings["final_two_batch_sizes"]

print("Training scans covered:", len(indices))
print("Batches per epoch:", len(check_batches))
print("Final two batch sizes:", final_sizes)
print("PASS | Every training row appears exactly once")

Training scans covered: 3889
Batches per epoch: 487
Final two batch sizes: [7, 2]
PASS | Every training row appears exactly once


In [8]:
from torch.utils.data import Dataset, DataLoader

class_to_index = protocol["class_to_index"]
checkpoint_normalize = T.Normalize(
    mean=(0.485, 0.456, 0.406),
    std=(0.229, 0.224, 0.225),
)

class TrainingEpochDataset(Dataset):
    def __init__(self, rows, run_seed, epoch):
        self.rows = rows
        self.run_seed = run_seed
        self.epoch = epoch

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, index):
        row = self.rows[index]
        image = load_canonical_mri(dataset_root / row["relative_path"])
        image = augment_training_image(image, self.run_seed, self.epoch, index)
        image = checkpoint_normalize(image)
        label = class_to_index[row["tumor_label"]]
        return image, label, index


preview_seed = protocol["seeds"][0]
preview_dataset = TrainingEpochDataset(records["train"], preview_seed, epoch=1)
preview_loader = DataLoader(
    preview_dataset,
    batch_sampler=check_batches,
    num_workers=protocol["training_loader"]["num_workers"],
    generator=torch.Generator().manual_seed(preview_seed),
)

images, labels, row_indices = next(iter(preview_loader))
expected_indices = check_batches[0]
expected_labels = [
    class_to_index[records["train"][i]["tumor_label"]]
    for i in expected_indices
]

assert row_indices.tolist() == expected_indices, "Batch order differs from Cell 7"
assert labels.tolist() == expected_labels, "Labels differ from the manifest"
assert tuple(images.shape) == (batch_size, 3, image_size, image_size)
assert images.dtype == torch.float32
assert torch.isfinite(images).all().item(), "Batch contains NaN or infinity"

print("Batch shape:", tuple(images.shape))
print("Manifest row indices:", row_indices.tolist())
print("Class indices:", labels.tolist())
print("PASS | First training batch matches the manifest")

Batch shape: (8, 3, 224, 224)
Manifest row indices: [3249, 996, 1152, 2663, 400, 1380, 867, 405]
Class indices: [3, 0, 0, 1, 0, 3, 3, 2]
PASS | First training batch matches the manifest


In [9]:
class ValidationDataset(Dataset):
    def __init__(self, rows):
        self.rows = rows

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, index):
        row = self.rows[index]
        image = load_canonical_mri(dataset_root / row["relative_path"])
        image = checkpoint_normalize(image)
        label = class_to_index[row["tumor_label"]]
        return image, label, index


validation_rows = records["validation"]
validation_loader = DataLoader(
    ValidationDataset(validation_rows),
    batch_size=batch_size,
    shuffle=False,
    drop_last=False,
    num_workers=0,
    generator=torch.Generator().manual_seed(preview_seed),
)

seen_indices = []
seen_labels = []
validation_batch_sizes = []

for val_images, val_labels, val_indices in validation_loader:
    count = len(val_indices)
    assert tuple(val_images.shape) == (count, 3, image_size, image_size)
    assert val_images.dtype == torch.float32
    assert torch.isfinite(val_images).all().item(), (
        f"Non-finite input in validation rows {val_indices.tolist()}"
    )

    seen_indices.extend(val_indices.tolist())
    seen_labels.extend(val_labels.tolist())
    validation_batch_sizes.append(count)

expected_labels = [class_to_index[row["tumor_label"]] for row in validation_rows]
assert seen_indices == list(range(len(validation_rows))), "Validation order differs"
assert seen_labels == expected_labels, "Validation labels differ from the manifest"
assert len(seen_indices) == protocol["validation"]["rows_per_epoch"]
assert validation_batch_sizes == [8] * 121 + [5], "Unexpected validation batch sizes"

print("Validation scans checked:", len(seen_indices))
print("Validation batches:", len(validation_batch_sizes))
print("Final batch size:", validation_batch_sizes[-1])
print("PASS | All validation scans loaded once, in order, with matching labels")

Validation scans checked: 973
Validation batches: 122
Final batch size: 5
PASS | All validation scans loaded once, in order, with matching labels


In [10]:
from huggingface_hub import hf_hub_download

model_id = "mobilenetv4_conv_small.e2400_r224_in1k"
pin = protocol["pretrained_checkpoints"][model_id]

weight_path = Path(hf_hub_download(
    repo_id=pin["repository"],
    revision=pin["revision"],
    filename=pin["filename"],
    local_files_only=True,
))

actual_hash = sha256(weight_path.read_bytes()).hexdigest()
if actual_hash != pin["sha256"]:
    raise ValueError(
        f"MobileNetV4 checkpoint differs from Notebook 08.\n"
        f"Expected: {pin['sha256']}\n"
        f"Found:    {actual_hash}"
    )

print("Model:", model_id)
print("Checkpoint file:", weight_path.name)
print("SHA-256:", actual_hash)
print("PASS | Pretrained MobileNetV4 checkpoint matches Notebook 08")

Model: mobilenetv4_conv_small.e2400_r224_in1k
Checkpoint file: model.safetensors
SHA-256: 7a7102ec18f62bbfb555b6fe829bbb5af749516b84174926c29ffdfdfc03aec4
PASS | Pretrained MobileNetV4 checkpoint matches Notebook 08


In [11]:
from safetensors.torch import load_file

device = torch.device("cuda:0")
run_seed = protocol["seeds"][0]
num_classes = len(class_to_index)

# Use Notebook 08's initialization so the new classifier is reproducible.
with torch.random.fork_rng(devices=[]):
    torch.default_generator.manual_seed(run_seed)
    model = timm.create_model(model_id, pretrained=False, num_classes=num_classes)

source = load_file(str(weight_path), device="cpu")
target = model.state_dict()
assert source.keys() == target.keys(), "Checkpoint and model tensor names differ"

fresh_names = []
for name, tensor in source.items():
    if tensor.shape == target[name].shape:
        target[name] = tensor
        continue

    # Only the classifier weight and bias should change from 1,000 to 4 outputs.
    old_shape = tuple(tensor.shape)
    new_shape = tuple(target[name].shape)
    assert old_shape[:1] == (1000,) and new_shape == (num_classes,) + old_shape[1:], (
        f"Unexpected shape change for {name}: {old_shape} -> {new_shape}"
    )
    fresh_names.append(name)

assert len(fresh_names) == 2, f"Unexpected classifier tensors: {fresh_names}"
model.load_state_dict(target, strict=True)
del source, target

model = model.to(device)
model.eval()
with torch.inference_mode():
    preview_scores = model(images.to(device))

assert preview_scores.shape == (len(images), num_classes)
assert torch.isfinite(preview_scores).all().item(), "Non-finite model scores"
assert all(p.requires_grad for p in model.parameters()), "Found a frozen parameter"

print("New classifier tensors:", ", ".join(fresh_names))
print("Trainable parameters:", f"{sum(p.numel() for p in model.parameters()):,}")
print("GPU output shape:", tuple(preview_scores.shape))
print("PASS | Four-class MobileNetV4 forward pass completed")

New classifier tensors: classifier.bias, classifier.weight
Trainable parameters: 2,498,148
GPU output shape: (8, 4)
PASS | Four-class MobileNetV4 forward pass completed


In [12]:
from torch import nn

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=protocol["learning_rate"],
    weight_decay=protocol["weight_decay"],
    foreach=protocol["optimizer_options"]["foreach"],
)

trainable_ids = {id(p) for p in model.parameters() if p.requires_grad}
optimizer_ids = [
    id(p) for group in optimizer.param_groups for p in group["params"]
]
assert set(optimizer_ids) == trainable_ids, "Optimizer parameter coverage differs"
assert len(optimizer_ids) == len(trainable_ids), "Repeated optimizer parameter"

group = optimizer.param_groups[0]
print("Loss:", type(criterion).__name__)
print("Optimizer:", type(optimizer).__name__)
print("Learning rate:", group["lr"])
print("Weight decay:", group["weight_decay"])
print("foreach:", group["foreach"])
print("PASS | Optimizer covers every trainable parameter exactly once")

Loss: CrossEntropyLoss
Optimizer: AdamW
Learning rate: 0.0001
Weight decay: 0.01
foreach: False
PASS | Optimizer covers every trainable parameter exactly once


In [13]:
from time import perf_counter

# A second execution would train the same model for another epoch.
if globals().get("timed_epoch_started", False):
    raise RuntimeError("This timed epoch has already started in this kernel.")

epoch = 1
torch.manual_seed(run_seed)

# Seed the shuffle generator once per run. Keep it for subsequent epochs.
shuffle_generator = torch.Generator().manual_seed(run_seed)
epoch_batches = make_training_batches(n_train, batch_size, shuffle_generator)
assert epoch_batches == check_batches, "Epoch 1 order differs from Cell 7"

train_dataset = TrainingEpochDataset(records["train"], run_seed, epoch)
train_loader = DataLoader(
    train_dataset,
    batch_sampler=epoch_batches,
    num_workers=protocol["training_loader"]["num_workers"],
    generator=torch.Generator().manual_seed(run_seed),
)

model.train()
loss_sum = torch.zeros((), device=device)
scans_seen = 0
timed_epoch_started = True

torch.cuda.synchronize(device)
start = perf_counter()

for batch_number, (inputs, targets, _) in enumerate(train_loader, start=1):
    inputs = inputs.to(device)
    targets = targets.to(device)

    optimizer.zero_grad(set_to_none=True)
    logits = model(inputs)
    loss = criterion(logits, targets)
    if not torch.isfinite(loss).item():
        raise FloatingPointError(f"Non-finite training loss in batch {batch_number}")

    loss.backward()
    optimizer.step()

    count = targets.size(0)
    loss_sum += loss.detach() * count
    scans_seen += count

    if batch_number % 100 == 0:
        print(f"Completed {batch_number}/{len(train_loader)} batches", flush=True)

torch.cuda.synchronize(device)
train_seconds = perf_counter() - start
assert scans_seen == n_train, f"Expected {n_train} scans, saw {scans_seen}"

train_loss = loss_sum.item() / scans_seen
print(f"Epoch {epoch} complete | {scans_seen:,} scans | {batch_number} batches")
print(f"Mean training loss: {train_loss:.4f}")
print(f"Training time: {train_seconds:.1f} seconds ({train_seconds / 60:.2f} min)")

Completed 100/487 batches
Completed 200/487 batches
Completed 300/487 batches
Completed 400/487 batches
Epoch 1 complete | 3,889 scans | 487 batches
Mean training loss: 1.5449
Training time: 61.5 seconds (1.02 min)


In [14]:
import torch.nn.functional as F

model.eval()
validation_loss_sum = 0.0
all_targets = []
all_predictions = []
validation_indices = []

torch.cuda.synchronize(device)
validation_start = perf_counter()

with torch.inference_mode():
    for inputs, targets, indices in validation_loader:
        inputs = inputs.to(device)
        targets = targets.to(device)
        logits = model(inputs)

        assert logits.shape == (len(targets), num_classes)
        assert torch.isfinite(logits).all().item(), (
            f"Non-finite validation scores at manifest rows {indices.tolist()}"
        )

        # Sum individual losses so the final five scans have the same
        # weight as every other scan when we calculate the mean.
        validation_loss_sum += F.cross_entropy(
            logits, targets, reduction="sum"
        ).item()
        all_targets.extend(targets.cpu().tolist())
        all_predictions.extend(logits.argmax(dim=1).cpu().tolist())
        validation_indices.extend(indices.tolist())

torch.cuda.synchronize(device)
validation_seconds = perf_counter() - validation_start

assert validation_indices == list(range(len(validation_rows))), (
    "Validation coverage or order differs from the manifest"
)

y_true = torch.tensor(all_targets)
y_pred = torch.tensor(all_predictions)
validation_loss = validation_loss_sum / len(y_true)
validation_accuracy = (y_true == y_pred).sum().item() / len(y_true)

per_class_f1 = {}
for name, index in class_to_index.items():
    actual = y_true == index
    predicted = y_pred == index
    true_positives = (actual & predicted).sum().item()
    denominator = actual.sum().item() + predicted.sum().item()
    per_class_f1[name] = 2 * true_positives / denominator if denominator else 0.0

validation_macro_f1 = sum(per_class_f1.values()) / num_classes

print(f"Epoch {epoch} validation | {len(y_true):,} scans")
print(f"Validation loss: {validation_loss:.4f}")
print(f"Validation accuracy: {validation_accuracy:.2%}")
for name, score in per_class_f1.items():
    print(f"  {name} F1: {score:.4f}")
print(f"Validation macro-F1: {validation_macro_f1:.6f}")
print(f"Validation time: {validation_seconds:.1f} seconds")

Epoch 1 validation | 973 scans
Validation loss: 0.5522
Validation accuracy: 83.56%
  glioma F1: 0.7771
  meningioma F1: 0.7364
  no_tumor F1: 0.9187
  pituitary F1: 0.8947
Validation macro-F1: 0.831686
Validation time: 3.6 seconds


In [15]:
from uuid import uuid4

checkpoint_dir = project_root / "outputs" / "checkpoints"
checkpoint_dir.mkdir(parents=True, exist_ok=True)

# A unique filename preserves earlier checkpoints if this cell is run again.
save_id = uuid4().hex[:12]
stem = f"mobilenetv4_conv_small_seed{run_seed}_epoch{epoch:02d}_{save_id}"
checkpoint_path = checkpoint_dir / f"{stem}.pt"
temporary_path = checkpoint_dir / f"{stem}.tmp"

checkpoint = {
    "model_id": model_id,
    "epoch": epoch,
    "run_seed": run_seed,
    "class_to_index": class_to_index,
    "model_state_dict": model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    "shuffle_generator_state": shuffle_generator.get_state(),
    "torch_cpu_rng_state": torch.get_rng_state(),
    "torch_cuda_rng_state": torch.cuda.get_rng_state(device),
    "protocol": protocol,
    "training_augmentation": aug_policy,
    "metrics": {
        "train_loss": train_loss,
        "validation_loss": validation_loss,
        "validation_accuracy": validation_accuracy,
        "validation_macro_f1": validation_macro_f1,
        "per_class_f1": per_class_f1,
        "train_seconds": train_seconds,
        "validation_seconds": validation_seconds,
    },
}

try:
    torch.save(checkpoint, temporary_path)
    saved = torch.load(temporary_path, map_location="cpu", weights_only=True)

    assert saved["metrics"]["validation_macro_f1"] == validation_macro_f1
    saved_weights = saved["model_state_dict"]
    live_weights = model.state_dict()
    assert saved_weights.keys() == live_weights.keys()
    for name, tensor in live_weights.items():
        assert torch.equal(saved_weights[name], tensor.cpu()), (
            f"Saved tensor differs: {name}"
        )

    temporary_path.replace(checkpoint_path)
finally:
    temporary_path.unlink(missing_ok=True)

print("Saved checkpoint:", checkpoint_path)
print("Stored validation macro-F1:", repr(saved["metrics"]["validation_macro_f1"]))
print("PASS | Epoch 1 checkpoint verified against the trained model")

Saved checkpoint: C:\Users\HP\Documents\Projects\Brain_Tumor_XAI\outputs\checkpoints\mobilenetv4_conv_small_seed42_epoch01_23d3efc4ac9e.pt
Stored validation macro-F1: 0.8316863817118774
PASS | Epoch 1 checkpoint verified against the trained model


In [16]:
def train_one_epoch(epoch_number):
    dataset = TrainingEpochDataset(records["train"], run_seed, epoch_number)
    batches = make_training_batches(len(dataset), batch_size, shuffle_generator)

    loader = DataLoader(
        dataset,
        batch_sampler=batches,
        num_workers=protocol["training_loader"]["num_workers"],
        generator=torch.Generator().manual_seed(run_seed),
    )

    model.train()
    loss_sum = torch.zeros((), device=device)
    scans_seen = 0

    torch.cuda.synchronize(device)
    start = perf_counter()

    for batch_number, (inputs, targets, _) in enumerate(loader, start=1):
        inputs = inputs.to(device)
        targets = targets.to(device)

        optimizer.zero_grad(set_to_none=True)
        loss = criterion(model(inputs), targets)
        if not torch.isfinite(loss).item():
            raise FloatingPointError(
                f"Non-finite loss at epoch {epoch_number}, batch {batch_number}"
            )

        loss.backward()
        optimizer.step()

        count = targets.size(0)
        loss_sum += loss.detach() * count
        scans_seen += count

    torch.cuda.synchronize(device)
    elapsed = perf_counter() - start
    assert scans_seen == len(dataset), f"Processed {scans_seen} scans"

    return loss_sum.item() / scans_seen, elapsed


print("Training function ready | MobileNetV4 | seed 42 | next epoch: 2")

Training function ready | MobileNetV4 | seed 42 | next epoch: 2


In [17]:
@torch.inference_mode()
def validate_one_epoch():
    model.eval()
    loss_sum = 0.0
    targets_seen = []
    predictions = []
    row_indices = []

    torch.cuda.synchronize(device)
    start = perf_counter()

    for inputs, targets, indices in validation_loader:
        inputs = inputs.to(device)
        targets = targets.to(device)
        logits = model(inputs)

        assert logits.shape == (len(targets), num_classes)
        assert torch.isfinite(logits).all().item(), (
            f"Non-finite validation scores at rows {indices.tolist()}"
        )

        loss_sum += F.cross_entropy(logits, targets, reduction="sum").item()
        targets_seen.extend(targets.cpu().tolist())
        predictions.extend(logits.argmax(dim=1).cpu().tolist())
        row_indices.extend(indices.tolist())

    torch.cuda.synchronize(device)
    elapsed = perf_counter() - start
    assert row_indices == list(range(len(validation_rows))), "Validation order differs"

    y_true = torch.tensor(targets_seen)
    y_pred = torch.tensor(predictions)
    per_class = {}

    for name, index in class_to_index.items():
        actual = y_true == index
        predicted = y_pred == index
        true_positives = (actual & predicted).sum().item()
        denominator = actual.sum().item() + predicted.sum().item()
        per_class[name] = 2 * true_positives / denominator if denominator else 0.0

    return {
        "validation_loss": loss_sum / len(y_true),
        "validation_accuracy": (y_true == y_pred).sum().item() / len(y_true),
        "validation_macro_f1": sum(per_class.values()) / num_classes,
        "per_class_f1": per_class,
        "validation_seconds": elapsed,
    }


validation_check = validate_one_epoch()
saved_f1 = saved["metrics"]["validation_macro_f1"]
checked_f1 = validation_check["validation_macro_f1"]

print("Saved epoch 1 macro-F1:", saved_f1)
print("Repeated macro-F1:     ", checked_f1)
assert checked_f1 == saved_f1, "Repeated validation differs from the saved checkpoint"
print("PASS | Validation function reproduces the saved epoch 1 macro-F1")

Saved epoch 1 macro-F1: 0.8316863817118774
Repeated macro-F1:      0.8316863817118774
PASS | Validation function reproduces the saved epoch 1 macro-F1


In [18]:
# Keep this run tied to the verified epoch 1 checkpoint from Cell 15.
run_dir = checkpoint_path.parent / checkpoint_path.stem
run_dir.mkdir(exist_ok=True)
last_epoch = protocol["epochs_per_run"]

# Find the latest consecutive checkpoint for this run.
resume_path = checkpoint_path
for number in range(2, last_epoch + 1):
    candidate = run_dir / f"epoch_{number:02d}.pt"
    if not candidate.is_file():
        break
    resume_path = candidate

resume = torch.load(resume_path, map_location="cpu", weights_only=True)
assert resume["model_id"] == model_id and resume["run_seed"] == run_seed
assert resume["protocol"] == protocol
assert resume["training_augmentation"] == aug_policy

model.load_state_dict(resume["model_state_dict"])
optimizer.load_state_dict(resume["optimizer_state_dict"])
shuffle_generator.set_state(resume["shuffle_generator_state"])
torch.set_rng_state(resume["torch_cpu_rng_state"])
torch.cuda.set_rng_state(resume["torch_cuda_rng_state"], device=device)

completed_epoch = resume["epoch"]
history = resume.get("history", {completed_epoch: resume["metrics"]})
del resume

# max() keeps the first entry on a tie; sorting puts earlier epochs first.
best_epoch = max(sorted(history), key=lambda n: history[n]["validation_macro_f1"])
best_f1 = history[best_epoch]["validation_macro_f1"]

print(f"Loaded epoch {completed_epoch} | best so far: epoch {best_epoch}", flush=True)

for number in range(completed_epoch + 1, last_epoch + 1):
    epoch_loss, epoch_seconds = train_one_epoch(number)
    metrics = validate_one_epoch()
    metrics["train_loss"] = epoch_loss
    metrics["train_seconds"] = epoch_seconds
    history[number] = metrics

    # Publish the final filename only after the checkpoint is fully written.
    epoch_path = run_dir / f"epoch_{number:02d}.pt"
    temp_path = epoch_path.with_suffix(".tmp")
    try:
        torch.save({
            "model_id": model_id,
            "epoch": number,
            "run_seed": run_seed,
            "class_to_index": class_to_index,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "shuffle_generator_state": shuffle_generator.get_state(),
            "torch_cpu_rng_state": torch.get_rng_state(),
            "torch_cuda_rng_state": torch.cuda.get_rng_state(device),
            "protocol": protocol,
            "training_augmentation": aug_policy,
            "metrics": metrics,
            "history": history,
        }, temp_path)
        temp_path.replace(epoch_path)
    finally:
        temp_path.unlink(missing_ok=True)

    score = metrics["validation_macro_f1"]
    improved = score > best_f1
    if improved:
        best_epoch, best_f1 = number, score

    print(
        f"Epoch {number:02d}/{last_epoch} | train loss {epoch_loss:.4f} | "
        f"val loss {metrics['validation_loss']:.4f} | "
        f"val acc {metrics['validation_accuracy']:.2%} | "
        f"macro-F1 {score:.6f} | saved"
        + (" | new best" if improved else ""),
        flush=True,
    )

best_path = checkpoint_path if best_epoch == 1 else run_dir / f"epoch_{best_epoch:02d}.pt"
print(f"\nBest epoch: {best_epoch}")
print(f"Best validation macro-F1: {best_f1!r}")
print(f"Selected checkpoint: {best_path}")

Loaded epoch 1 | best so far: epoch 1
Epoch 02/15 | train loss 0.6125 | val loss 0.3452 | val acc 88.28% | macro-F1 0.880513 | saved | new best
Epoch 03/15 | train loss 0.5333 | val loss 0.2839 | val acc 89.62% | macro-F1 0.896817 | saved | new best
Epoch 04/15 | train loss 0.3569 | val loss 0.3185 | val acc 90.03% | macro-F1 0.900299 | saved | new best
Epoch 05/15 | train loss 0.3656 | val loss 0.2135 | val acc 92.70% | macro-F1 0.927855 | saved | new best
Epoch 06/15 | train loss 0.2762 | val loss 0.5441 | val acc 87.15% | macro-F1 0.876089 | saved
Epoch 07/15 | train loss 0.2685 | val loss 1.2974 | val acc 80.47% | macro-F1 0.816329 | saved
Epoch 08/15 | train loss 0.2745 | val loss 0.3825 | val acc 89.11% | macro-F1 0.893883 | saved
Epoch 09/15 | train loss 0.2381 | val loss 0.4677 | val acc 89.62% | macro-F1 0.896585 | saved
Epoch 10/15 | train loss 0.2851 | val loss 0.2251 | val acc 92.60% | macro-F1 0.925104 | saved
Epoch 11/15 | train loss 0.1964 | val loss 1.1990 | val acc 86.

In [19]:
selected = torch.load(best_path, map_location="cpu", weights_only=True)
assert selected["model_id"] == model_id
assert selected["run_seed"] == run_seed
assert selected["epoch"] == best_epoch
assert selected["class_to_index"] == class_to_index

model.load_state_dict(selected["model_state_dict"], strict=True)
verified = validate_one_epoch()
stored = selected["metrics"]

print("Selected checkpoint:", best_path.name)
print("Stored macro-F1:    ", stored["validation_macro_f1"])
print("Recomputed macro-F1:", verified["validation_macro_f1"])

assert verified["validation_macro_f1"] == stored["validation_macro_f1"] == best_f1
assert verified["validation_accuracy"] == stored["validation_accuracy"]
assert verified["per_class_f1"] == stored["per_class_f1"]

print(f"\nEpoch {best_epoch} validation accuracy: {verified['validation_accuracy']:.2%}")
for name, score in verified["per_class_f1"].items():
    print(f"  {name} F1: {score:.4f}")
print("PASS | Selected checkpoint reproduces the recorded validation scores")

Selected checkpoint: epoch_14.pt
Stored macro-F1:     0.9547767162770389
Recomputed macro-F1: 0.9547767162770389

Epoch 14 validation accuracy: 95.48%
  glioma F1: 0.9451
  meningioma F1: 0.9361
  no_tumor F1: 0.9742
  pituitary F1: 0.9637
PASS | Selected checkpoint reproduces the recorded validation scores


In [20]:
convnext_id = "convnextv2_atto.fcmae_ft_in1k"
convnext_pin = protocol["pretrained_checkpoints"][convnext_id]

convnext_weight_path = Path(hf_hub_download(
    repo_id=convnext_pin["repository"],
    revision=convnext_pin["revision"],
    filename=convnext_pin["filename"],
    local_files_only=True,
))

convnext_hash = sha256(convnext_weight_path.read_bytes()).hexdigest()
if convnext_hash != convnext_pin["sha256"]:
    raise ValueError(
        f"ConvNeXt checkpoint differs from Notebook 08.\n"
        f"Expected: {convnext_pin['sha256']}\n"
        f"Found:    {convnext_hash}"
    )

print("Model:", convnext_id)
print("Checkpoint file:", convnext_weight_path.name)
print("SHA-256:", convnext_hash)
print("PASS | Pretrained ConvNeXt V2 Atto checkpoint matches Notebook 08")

Model: convnextv2_atto.fcmae_ft_in1k
Checkpoint file: model.safetensors
SHA-256: 6389c2f5a427b01a922e66e6d352c707424cccb62390c6936bc612e3d10b7ebb
PASS | Pretrained ConvNeXt V2 Atto checkpoint matches Notebook 08


In [21]:
convnext_seed = 42

with torch.random.fork_rng(devices=[]):
    torch.default_generator.manual_seed(convnext_seed)
    convnext_model = timm.create_model(
        convnext_id, pretrained=False, num_classes=num_classes
    )

source_weights = load_file(str(convnext_weight_path), device="cpu")
target_weights = convnext_model.state_dict()
assert source_weights.keys() == target_weights.keys(), "Tensor names differ"

fresh_names = []
for name, source_tensor in source_weights.items():
    target_tensor = target_weights[name]
    if source_tensor.shape == target_tensor.shape:
        target_weights[name] = source_tensor
        continue

    # Only the final classifier changes from 1,000 outputs to four.
    old_shape = tuple(source_tensor.shape)
    new_shape = tuple(target_tensor.shape)
    assert old_shape[:1] == (1000,) and new_shape == (num_classes,) + old_shape[1:], (
        f"Unexpected shape change for {name}: {old_shape} -> {new_shape}"
    )
    fresh_names.append(name)

assert len(fresh_names) == 2, f"Unexpected classifier tensors: {fresh_names}"
convnext_model.load_state_dict(target_weights, strict=True)
del source_weights, target_weights

convnext_model = convnext_model.to(device)
assert all(p.requires_grad for p in convnext_model.parameters())
trainable_count = sum(p.numel() for p in convnext_model.parameters())
assert trainable_count == 3_388_684, f"Unexpected parameter count: {trainable_count}"

convnext_optimizer = torch.optim.AdamW(
    convnext_model.parameters(),
    lr=protocol["learning_rate"],
    weight_decay=protocol["weight_decay"],
    foreach=protocol["optimizer_options"]["foreach"],
)

convnext_model.eval()
with torch.inference_mode():
    convnext_scores = convnext_model(images.to(device))

assert convnext_scores.shape == (len(images), num_classes)
assert torch.isfinite(convnext_scores).all().item(), "Non-finite model scores"

print("New classifier tensors:", ", ".join(fresh_names))
print("Trainable parameters:", f"{trainable_count:,}")
print("GPU output shape:", tuple(convnext_scores.shape))
print("PASS | Four-class ConvNeXt and fresh AdamW optimizer are ready")

New classifier tensors: head.fc.bias, head.fc.weight
Trainable parameters: 3,388,684
GPU output shape: (8, 4)
PASS | Four-class ConvNeXt and fresh AdamW optimizer are ready


In [22]:
def train_model_epoch(active_model, active_optimizer, seed, epoch_number, batches):
    dataset = TrainingEpochDataset(records["train"], seed, epoch_number)
    loader = DataLoader(
        dataset,
        batch_sampler=batches,
        num_workers=protocol["training_loader"]["num_workers"],
        generator=torch.Generator().manual_seed(seed),
    )

    active_model.train()
    loss_sum = torch.zeros((), device=device)
    scans_seen = 0

    torch.cuda.synchronize(device)
    start = perf_counter()

    for batch_number, (inputs, targets, _) in enumerate(loader, start=1):
        inputs = inputs.to(device)
        targets = targets.to(device)

        active_optimizer.zero_grad(set_to_none=True)
        logits = active_model(inputs)
        loss = criterion(logits, targets)
        if not torch.isfinite(loss).item():
            raise FloatingPointError(
                f"Non-finite loss at epoch {epoch_number}, batch {batch_number}"
            )

        loss.backward()
        active_optimizer.step()

        count = targets.size(0)
        loss_sum += loss.detach() * count
        scans_seen += count

        if batch_number % 100 == 0:
            print(f"Completed {batch_number}/{len(loader)} batches", flush=True)

    torch.cuda.synchronize(device)
    elapsed = perf_counter() - start
    assert scans_seen == len(dataset), f"Processed {scans_seen} scans"

    return loss_sum.item() / scans_seen, elapsed


if globals().get("convnext_epoch1_started", False):
    raise RuntimeError("ConvNeXt epoch 1 has already started in this kernel.")

torch.manual_seed(convnext_seed)
convnext_shuffle = torch.Generator().manual_seed(convnext_seed)
convnext_batches = make_training_batches(n_train, batch_size, convnext_shuffle)
assert convnext_batches == check_batches, "Seed-42 batch order differs"

convnext_epoch1_started = True
convnext_train_loss, convnext_train_seconds = train_model_epoch(
    convnext_model, convnext_optimizer, convnext_seed, 1, convnext_batches
)

print(f"ConvNeXt epoch 1 complete | {n_train:,} scans | {len(convnext_batches)} batches")
print(f"Mean training loss: {convnext_train_loss:.4f}")
print(f"Training time: {convnext_train_seconds:.1f} seconds")

Completed 100/487 batches
Completed 200/487 batches
Completed 300/487 batches
Completed 400/487 batches
ConvNeXt epoch 1 complete | 3,889 scans | 487 batches
Mean training loss: 0.3232
Training time: 81.8 seconds


In [23]:
@torch.inference_mode()
def validate_model(active_model):
    active_model.eval()
    loss_sum = 0.0
    targets_seen = []
    predictions = []
    row_indices = []

    torch.cuda.synchronize(device)
    start = perf_counter()

    for inputs, targets, indices in validation_loader:
        inputs = inputs.to(device)
        targets = targets.to(device)
        logits = active_model(inputs)

        assert logits.shape == (len(targets), num_classes)
        assert torch.isfinite(logits).all().item(), (
            f"Non-finite validation scores at rows {indices.tolist()}"
        )

        loss_sum += F.cross_entropy(logits, targets, reduction="sum").item()
        targets_seen.extend(targets.cpu().tolist())
        predictions.extend(logits.argmax(dim=1).cpu().tolist())
        row_indices.extend(indices.tolist())

    torch.cuda.synchronize(device)
    elapsed = perf_counter() - start
    assert row_indices == list(range(len(validation_rows))), "Validation order differs"

    y_true = torch.tensor(targets_seen)
    y_pred = torch.tensor(predictions)
    per_class = {}

    for name, index in class_to_index.items():
        actual = y_true == index
        predicted = y_pred == index
        true_positives = (actual & predicted).sum().item()
        denominator = actual.sum().item() + predicted.sum().item()
        per_class[name] = 2 * true_positives / denominator if denominator else 0.0

    return {
        "validation_loss": loss_sum / len(y_true),
        "validation_accuracy": (y_true == y_pred).sum().item() / len(y_true),
        "validation_macro_f1": sum(per_class.values()) / num_classes,
        "per_class_f1": per_class,
        "validation_seconds": elapsed,
    }


convnext_metrics = validate_model(convnext_model)
convnext_metrics["train_loss"] = convnext_train_loss
convnext_metrics["train_seconds"] = convnext_train_seconds

# Keep all checkpoints for this ConvNeXt run in their own folder.
if "convnext_run_dir" not in globals():
    convnext_run_dir = checkpoint_dir / f"convnextv2_atto_seed42_{uuid4().hex[:12]}"
convnext_run_dir.mkdir(parents=True, exist_ok=True)

convnext_epoch1_path = convnext_run_dir / "epoch_01.pt"
temp_path = convnext_epoch1_path.with_suffix(".tmp")

try:
    torch.save({
        "model_id": convnext_id,
        "epoch": 1,
        "run_seed": convnext_seed,
        "class_to_index": class_to_index,
        "model_state_dict": convnext_model.state_dict(),
        "optimizer_state_dict": convnext_optimizer.state_dict(),
        "shuffle_generator_state": convnext_shuffle.get_state(),
        "torch_cpu_rng_state": torch.get_rng_state(),
        "torch_cuda_rng_state": torch.cuda.get_rng_state(device),
        "protocol": protocol,
        "training_augmentation": aug_policy,
        "metrics": convnext_metrics,
        "history": {1: convnext_metrics},
    }, temp_path)

    restored = torch.load(temp_path, map_location="cpu", weights_only=True)
    assert restored["metrics"] == convnext_metrics
    for name, tensor in convnext_model.state_dict().items():
        assert torch.equal(restored["model_state_dict"][name], tensor.cpu()), name
    temp_path.replace(convnext_epoch1_path)
finally:
    temp_path.unlink(missing_ok=True)

print(f"ConvNeXt epoch 1 validation | {len(validation_rows)} scans")
print(f"Validation loss: {convnext_metrics['validation_loss']:.4f}")
print(f"Validation accuracy: {convnext_metrics['validation_accuracy']:.2%}")
for name, score in convnext_metrics["per_class_f1"].items():
    print(f"  {name} F1: {score:.4f}")
print("Validation macro-F1:", convnext_metrics["validation_macro_f1"])
print(f"Validation time: {convnext_metrics['validation_seconds']:.1f} seconds")
print("PASS | Checkpoint verified and saved:", convnext_epoch1_path)

ConvNeXt epoch 1 validation | 973 scans
Validation loss: 0.1775
Validation accuracy: 93.94%
  glioma F1: 0.9198
  meningioma F1: 0.8918
  no_tumor F1: 0.9758
  pituitary F1: 0.9664
Validation macro-F1: 0.9384736197629343
Validation time: 5.8 seconds
PASS | Checkpoint verified and saved: C:\Users\HP\Documents\Projects\Brain_Tumor_XAI\outputs\checkpoints\convnextv2_atto_seed42_516f51d37e0b\epoch_01.pt


In [24]:
final_epoch = protocol["epochs_per_run"]
resume_path = convnext_epoch1_path

for number in range(2, final_epoch + 1):
    candidate = convnext_run_dir / f"epoch_{number:02d}.pt"
    if not candidate.is_file():
        break
    resume_path = candidate

resume = torch.load(resume_path, map_location="cpu", weights_only=True)
assert resume["model_id"] == convnext_id
assert resume["run_seed"] == convnext_seed
assert resume["protocol"] == protocol
assert resume["training_augmentation"] == aug_policy
assert resume["class_to_index"] == class_to_index

convnext_model.load_state_dict(resume["model_state_dict"], strict=True)
convnext_optimizer.load_state_dict(resume["optimizer_state_dict"])
convnext_shuffle.set_state(resume["shuffle_generator_state"])
torch.set_rng_state(resume["torch_cpu_rng_state"])
torch.cuda.set_rng_state(resume["torch_cuda_rng_state"], device=device)

completed_epoch = resume["epoch"]
convnext_history = resume["history"]
del resume

convnext_best_epoch = max(
    sorted(convnext_history),
    key=lambda n: convnext_history[n]["validation_macro_f1"],
)
convnext_best_f1 = convnext_history[convnext_best_epoch]["validation_macro_f1"]
print(
    f"Loaded ConvNeXt epoch {completed_epoch} | best so far: "
    f"epoch {convnext_best_epoch}, macro-F1 {convnext_best_f1:.6f}",
    flush=True,
)

for number in range(completed_epoch + 1, final_epoch + 1):
    batches = make_training_batches(n_train, batch_size, convnext_shuffle)
    train_loss, train_seconds = train_model_epoch(
        convnext_model, convnext_optimizer, convnext_seed, number, batches
    )
    metrics = validate_model(convnext_model)
    metrics["train_loss"] = train_loss
    metrics["train_seconds"] = train_seconds
    convnext_history[number] = metrics

    epoch_path = convnext_run_dir / f"epoch_{number:02d}.pt"
    temp_path = epoch_path.with_suffix(".tmp")
    try:
        torch.save({
            "model_id": convnext_id,
            "epoch": number,
            "run_seed": convnext_seed,
            "class_to_index": class_to_index,
            "model_state_dict": convnext_model.state_dict(),
            "optimizer_state_dict": convnext_optimizer.state_dict(),
            "shuffle_generator_state": convnext_shuffle.get_state(),
            "torch_cpu_rng_state": torch.get_rng_state(),
            "torch_cuda_rng_state": torch.cuda.get_rng_state(device),
            "protocol": protocol,
            "training_augmentation": aug_policy,
            "metrics": metrics,
            "history": convnext_history,
        }, temp_path)
        temp_path.replace(epoch_path)
    finally:
        temp_path.unlink(missing_ok=True)

    score = metrics["validation_macro_f1"]
    improved = score > convnext_best_f1
    if improved:
        convnext_best_epoch, convnext_best_f1 = number, score

    print(
        f"Epoch {number:02d}/{final_epoch} | train loss {train_loss:.4f} | "
        f"val loss {metrics['validation_loss']:.4f} | "
        f"val acc {metrics['validation_accuracy']:.2%} | "
        f"macro-F1 {score:.6f} | saved"
        + (" | new best" if improved else ""),
        flush=True,
    )

convnext_best_path = convnext_run_dir / f"epoch_{convnext_best_epoch:02d}.pt"
print(f"\nBest epoch: {convnext_best_epoch}")
print(f"Best validation macro-F1: {convnext_best_f1!r}")
print(f"Selected checkpoint: {convnext_best_path}")

Loaded ConvNeXt epoch 1 | best so far: epoch 1, macro-F1 0.938474
Completed 100/487 batches
Completed 200/487 batches
Completed 300/487 batches
Completed 400/487 batches
Epoch 02/15 | train loss 0.1263 | val loss 0.1214 | val acc 96.30% | macro-F1 0.962219 | saved | new best
Completed 100/487 batches
Completed 200/487 batches
Completed 300/487 batches
Completed 400/487 batches
Epoch 03/15 | train loss 0.0845 | val loss 0.0541 | val acc 97.94% | macro-F1 0.979247 | saved | new best
Completed 100/487 batches
Completed 200/487 batches
Completed 300/487 batches
Completed 400/487 batches
Epoch 04/15 | train loss 0.0851 | val loss 0.1827 | val acc 93.73% | macro-F1 0.937224 | saved
Completed 100/487 batches
Completed 200/487 batches
Completed 300/487 batches
Completed 400/487 batches
Epoch 05/15 | train loss 0.0512 | val loss 0.0806 | val acc 97.12% | macro-F1 0.971073 | saved
Completed 100/487 batches
Completed 200/487 batches
Completed 300/487 batches
Completed 400/487 batches
Epoch 06/15 

In [25]:
convnext_selected = torch.load(
    convnext_best_path, map_location="cpu", weights_only=True
)

assert convnext_selected["model_id"] == convnext_id
assert convnext_selected["run_seed"] == convnext_seed
assert convnext_selected["epoch"] == convnext_best_epoch
assert convnext_selected["class_to_index"] == class_to_index

convnext_model.load_state_dict(convnext_selected["model_state_dict"], strict=True)
convnext_verified = validate_model(convnext_model)
convnext_stored = convnext_selected["metrics"]

print("Selected checkpoint:", convnext_best_path.name)
print("Stored macro-F1:    ", convnext_stored["validation_macro_f1"])
print("Recomputed macro-F1:", convnext_verified["validation_macro_f1"])

assert convnext_verified["validation_macro_f1"] == convnext_stored["validation_macro_f1"]
assert convnext_verified["validation_macro_f1"] == convnext_best_f1
assert convnext_verified["validation_accuracy"] == convnext_stored["validation_accuracy"]
assert convnext_verified["per_class_f1"] == convnext_stored["per_class_f1"]

print(f"Validation accuracy: {convnext_verified['validation_accuracy']:.2%}")
for name, score in convnext_verified["per_class_f1"].items():
    print(f"  {name} F1: {score:.4f}")
print("PASS | Selected ConvNeXt checkpoint reproduces its validation scores")

Selected checkpoint: epoch_10.pt
Stored macro-F1:     0.9866446047733788
Recomputed macro-F1: 0.9866446047733788
Validation accuracy: 98.66%
  glioma F1: 0.9828
  meningioma F1: 0.9813
  no_tumor F1: 0.9928
  pituitary F1: 0.9897
PASS | Selected ConvNeXt checkpoint reproduces its validation scores


In [26]:
repvit_id = "repvit_m0_9.dist_450e_in1k"
repvit_pin = protocol["pretrained_checkpoints"][repvit_id]

repvit_weight_path = Path(hf_hub_download(
    repo_id=repvit_pin["repository"],
    revision=repvit_pin["revision"],
    filename=repvit_pin["filename"],
    local_files_only=True,
))

repvit_hash = sha256(repvit_weight_path.read_bytes()).hexdigest()
if repvit_hash != repvit_pin["sha256"]:
    raise ValueError(
        f"RepViT checkpoint differs from Notebook 08.\n"
        f"Expected: {repvit_pin['sha256']}\n"
        f"Found:    {repvit_hash}"
    )

print("Model:", repvit_id)
print("Checkpoint file:", repvit_weight_path.name)
print("SHA-256:", repvit_hash)
print("PASS | Pretrained RepViT M0.9 checkpoint matches Notebook 08")

Model: repvit_m0_9.dist_450e_in1k
Checkpoint file: model.safetensors
SHA-256: c2dd32622df856ea7a68d6b241c1623f0e6fbec8acde381a3519cb5a5463e9cf
PASS | Pretrained RepViT M0.9 checkpoint matches Notebook 08


In [27]:
repvit_seed = 42

with torch.random.fork_rng(devices=[]):
    torch.default_generator.manual_seed(repvit_seed)
    repvit_model = timm.create_model(
        repvit_id, pretrained=False, num_classes=num_classes
    )

source_weights = load_file(str(repvit_weight_path), device="cpu")
target_weights = repvit_model.state_dict()
assert source_weights.keys() == target_weights.keys(), "Tensor names differ"

repvit_fresh_names = []
for name, source_tensor in source_weights.items():
    target_tensor = target_weights[name]
    if source_tensor.shape == target_tensor.shape:
        target_weights[name] = source_tensor
        continue

    # Only the output dimension of the two classifiers may change.
    old_shape = tuple(source_tensor.shape)
    new_shape = tuple(target_tensor.shape)
    assert old_shape[:1] == (1000,) and new_shape == (num_classes,) + old_shape[1:], (
        f"Unexpected shape change for {name}: {old_shape} -> {new_shape}"
    )
    repvit_fresh_names.append(name)

assert len(repvit_fresh_names) == 4, f"Unexpected head tensors: {repvit_fresh_names}"
repvit_model.load_state_dict(target_weights, strict=True)
del source_weights, target_weights

repvit_model = repvit_model.to(device)
assert all(p.requires_grad for p in repvit_model.parameters())
repvit_count = sum(p.numel() for p in repvit_model.parameters())
assert repvit_count == 4_722_408, f"Unexpected parameter count: {repvit_count}"

repvit_optimizer = torch.optim.AdamW(
    repvit_model.parameters(),
    lr=protocol["learning_rate"],
    weight_decay=protocol["weight_decay"],
    foreach=protocol["optimizer_options"]["foreach"],
)

repvit_model.eval()
with torch.inference_mode():
    repvit_scores = repvit_model(images.to(device))

assert repvit_scores.shape == (len(images), num_classes)
assert torch.isfinite(repvit_scores).all().item(), "Non-finite model scores"

print("New classifier tensors:", ", ".join(repvit_fresh_names))
print("Trainable parameters:", f"{repvit_count:,}")
print("GPU output shape:", tuple(repvit_scores.shape))
print("PASS | Four-class RepViT and fresh AdamW optimizer are ready")

New classifier tensors: head.head.l.bias, head.head.l.weight, head.head_dist.l.bias, head.head_dist.l.weight
Trainable parameters: 4,722,408
GPU output shape: (8, 4)
PASS | Four-class RepViT and fresh AdamW optimizer are ready


In [28]:
if globals().get("repvit_epoch1_started", False):
    raise RuntimeError("RepViT epoch 1 has already started in this kernel.")

torch.manual_seed(repvit_seed)
repvit_shuffle = torch.Generator().manual_seed(repvit_seed)
repvit_batches = make_training_batches(n_train, batch_size, repvit_shuffle)
assert repvit_batches == check_batches, "Seed-42 batch order differs"

repvit_epoch1_started = True
repvit_train_loss, repvit_train_seconds = train_model_epoch(
    repvit_model, repvit_optimizer, repvit_seed, 1, repvit_batches
)

print(f"RepViT epoch 1 complete | {n_train:,} scans | {len(repvit_batches)} batches")
print(f"Mean training loss: {repvit_train_loss:.4f}")
print(f"Training time: {repvit_train_seconds:.1f} seconds")

Completed 100/487 batches
Completed 200/487 batches
Completed 300/487 batches
Completed 400/487 batches
RepViT epoch 1 complete | 3,889 scans | 487 batches
Mean training loss: 0.5091
Training time: 79.7 seconds


In [29]:
repvit_metrics = validate_model(repvit_model)
repvit_metrics["train_loss"] = repvit_train_loss
repvit_metrics["train_seconds"] = repvit_train_seconds

if "repvit_run_dir" not in globals():
    repvit_run_dir = checkpoint_dir / f"repvit_m0_9_seed42_{uuid4().hex[:12]}"
repvit_run_dir.mkdir(parents=True, exist_ok=True)

repvit_epoch1_path = repvit_run_dir / "epoch_01.pt"
temp_path = repvit_epoch1_path.with_suffix(".tmp")

try:
    torch.save({
        "model_id": repvit_id,
        "epoch": 1,
        "run_seed": repvit_seed,
        "class_to_index": class_to_index,
        "model_state_dict": repvit_model.state_dict(),
        "optimizer_state_dict": repvit_optimizer.state_dict(),
        "shuffle_generator_state": repvit_shuffle.get_state(),
        "torch_cpu_rng_state": torch.get_rng_state(),
        "torch_cuda_rng_state": torch.cuda.get_rng_state(device),
        "protocol": protocol,
        "training_augmentation": aug_policy,
        "metrics": repvit_metrics,
        "history": {1: repvit_metrics},
    }, temp_path)

    restored = torch.load(temp_path, map_location="cpu", weights_only=True)
    assert restored["metrics"] == repvit_metrics, "Saved metrics differ"
    for name, tensor in repvit_model.state_dict().items():
        assert torch.equal(restored["model_state_dict"][name], tensor.cpu()), name

    temp_path.replace(repvit_epoch1_path)
finally:
    temp_path.unlink(missing_ok=True)

print(f"RepViT epoch 1 validation | {len(validation_rows)} scans")
print(f"Validation loss: {repvit_metrics['validation_loss']:.4f}")
print(f"Validation accuracy: {repvit_metrics['validation_accuracy']:.2%}")
for name, score in repvit_metrics["per_class_f1"].items():
    print(f"  {name} F1: {score:.4f}")
print("Validation macro-F1:", repvit_metrics["validation_macro_f1"])
print(f"Validation time: {repvit_metrics['validation_seconds']:.1f} seconds")
print("PASS | Checkpoint verified and saved:", repvit_epoch1_path)

RepViT epoch 1 validation | 973 scans
Validation loss: 0.1695
Validation accuracy: 94.66%
  glioma F1: 0.9389
  meningioma F1: 0.9024
  no_tumor F1: 0.9858
  pituitary F1: 0.9590
Validation macro-F1: 0.946556727690542
Validation time: 4.5 seconds
PASS | Checkpoint verified and saved: C:\Users\HP\Documents\Projects\Brain_Tumor_XAI\outputs\checkpoints\repvit_m0_9_seed42_d8861b4c7f0e\epoch_01.pt


In [30]:
final_epoch = protocol["epochs_per_run"]
repvit_resume_path = repvit_epoch1_path

for number in range(2, final_epoch + 1):
    candidate = repvit_run_dir / f"epoch_{number:02d}.pt"
    if not candidate.is_file():
        break
    repvit_resume_path = candidate

resume = torch.load(repvit_resume_path, map_location="cpu", weights_only=True)
assert resume["model_id"] == repvit_id
assert resume["run_seed"] == repvit_seed
assert resume["protocol"] == protocol
assert resume["training_augmentation"] == aug_policy
assert resume["class_to_index"] == class_to_index

repvit_model.load_state_dict(resume["model_state_dict"], strict=True)
repvit_optimizer.load_state_dict(resume["optimizer_state_dict"])
repvit_shuffle.set_state(resume["shuffle_generator_state"])
torch.set_rng_state(resume["torch_cpu_rng_state"])
torch.cuda.set_rng_state(resume["torch_cuda_rng_state"], device=device)

completed_epoch = resume["epoch"]
repvit_history = resume["history"]
del resume

repvit_best_epoch = max(
    sorted(repvit_history),
    key=lambda n: repvit_history[n]["validation_macro_f1"],
)
repvit_best_f1 = repvit_history[repvit_best_epoch]["validation_macro_f1"]
print(
    f"Loaded RepViT epoch {completed_epoch} | best so far: "
    f"epoch {repvit_best_epoch}, macro-F1 {repvit_best_f1:.6f}",
    flush=True,
)

for number in range(completed_epoch + 1, final_epoch + 1):
    batches = make_training_batches(n_train, batch_size, repvit_shuffle)
    train_loss, train_seconds = train_model_epoch(
        repvit_model, repvit_optimizer, repvit_seed, number, batches
    )
    metrics = validate_model(repvit_model)
    metrics["train_loss"] = train_loss
    metrics["train_seconds"] = train_seconds
    repvit_history[number] = metrics

    epoch_path = repvit_run_dir / f"epoch_{number:02d}.pt"
    temp_path = epoch_path.with_suffix(".tmp")
    try:
        torch.save({
            "model_id": repvit_id,
            "epoch": number,
            "run_seed": repvit_seed,
            "class_to_index": class_to_index,
            "model_state_dict": repvit_model.state_dict(),
            "optimizer_state_dict": repvit_optimizer.state_dict(),
            "shuffle_generator_state": repvit_shuffle.get_state(),
            "torch_cpu_rng_state": torch.get_rng_state(),
            "torch_cuda_rng_state": torch.cuda.get_rng_state(device),
            "protocol": protocol,
            "training_augmentation": aug_policy,
            "metrics": metrics,
            "history": repvit_history,
        }, temp_path)
        temp_path.replace(epoch_path)
    finally:
        temp_path.unlink(missing_ok=True)

    score = metrics["validation_macro_f1"]
    improved = score > repvit_best_f1
    if improved:
        repvit_best_epoch, repvit_best_f1 = number, score

    print(
        f"Epoch {number:02d}/{final_epoch} | train loss {train_loss:.4f} | "
        f"val loss {metrics['validation_loss']:.4f} | "
        f"val acc {metrics['validation_accuracy']:.2%} | "
        f"macro-F1 {score:.6f} | saved"
        + (" | new best" if improved else ""),
        flush=True,
    )

repvit_best_path = repvit_run_dir / f"epoch_{repvit_best_epoch:02d}.pt"
print(f"\nBest epoch: {repvit_best_epoch}")
print(f"Best validation macro-F1: {repvit_best_f1!r}")
print(f"Selected checkpoint: {repvit_best_path}")

Loaded RepViT epoch 1 | best so far: epoch 1, macro-F1 0.946557
Completed 100/487 batches
Completed 200/487 batches
Completed 300/487 batches
Completed 400/487 batches
Epoch 02/15 | train loss 0.1926 | val loss 0.1207 | val acc 97.12% | macro-F1 0.972074 | saved | new best
Completed 100/487 batches
Completed 200/487 batches
Completed 300/487 batches
Completed 400/487 batches
Epoch 03/15 | train loss 0.1408 | val loss 0.1173 | val acc 96.81% | macro-F1 0.968905 | saved
Completed 100/487 batches
Completed 200/487 batches
Completed 300/487 batches
Completed 400/487 batches
Epoch 04/15 | train loss 0.0849 | val loss 0.0830 | val acc 97.64% | macro-F1 0.976665 | saved | new best
Completed 100/487 batches
Completed 200/487 batches
Completed 300/487 batches
Completed 400/487 batches
Epoch 05/15 | train loss 0.0728 | val loss 0.0630 | val acc 98.15% | macro-F1 0.981764 | saved | new best
Completed 100/487 batches
Completed 200/487 batches
Completed 300/487 batches
Completed 400/487 batches
Epo

In [31]:
repvit_selected = torch.load(
    repvit_best_path, map_location="cpu", weights_only=True
)

assert repvit_selected["model_id"] == repvit_id
assert repvit_selected["run_seed"] == repvit_seed
assert repvit_selected["epoch"] == repvit_best_epoch
assert repvit_selected["class_to_index"] == class_to_index

repvit_model.load_state_dict(repvit_selected["model_state_dict"], strict=True)
repvit_verified = validate_model(repvit_model)
repvit_stored = repvit_selected["metrics"]

print("Selected checkpoint:", repvit_best_path.name)
print("Stored macro-F1:    ", repvit_stored["validation_macro_f1"])
print("Recomputed macro-F1:", repvit_verified["validation_macro_f1"])

assert repvit_verified["validation_macro_f1"] == repvit_stored["validation_macro_f1"]
assert repvit_verified["validation_macro_f1"] == repvit_best_f1
assert repvit_verified["validation_accuracy"] == repvit_stored["validation_accuracy"]
assert repvit_verified["per_class_f1"] == repvit_stored["per_class_f1"]

print(f"Validation accuracy: {repvit_verified['validation_accuracy']:.2%}")
for name, score in repvit_verified["per_class_f1"].items():
    print(f"  {name} F1: {score:.4f}")
print("PASS | Selected RepViT checkpoint reproduces its validation scores")

Selected checkpoint: epoch_15.pt
Stored macro-F1:     0.9929969375289072
Recomputed macro-F1: 0.9929969375289072
Validation accuracy: 99.28%
  glioma F1: 0.9957
  meningioma F1: 0.9897
  no_tumor F1: 0.9952
  pituitary F1: 0.9914
PASS | Selected RepViT checkpoint reproduces its validation scores


In [32]:
import csv

seed42_runs = [
    ("RepViT M0.9", repvit_selected, repvit_best_path),
    ("ConvNeXt V2 Atto", convnext_selected, convnext_best_path),
    ("MobileNetV4 Conv Small", selected, best_path),
]

summary_rows = []
for model_name, checkpoint_data, path in seed42_runs:
    assert checkpoint_data["run_seed"] == 42
    metrics = checkpoint_data["metrics"]

    row = {
        "model": model_name,
        "model_id": checkpoint_data["model_id"],
        "seed": checkpoint_data["run_seed"],
        "selected_epoch": checkpoint_data["epoch"],
        "validation_scans": len(validation_rows),
        "validation_accuracy": metrics["validation_accuracy"],
        "validation_macro_f1": metrics["validation_macro_f1"],
        "validation_loss": metrics["validation_loss"],
    }
    for label in class_to_index:
        row[f"f1_{label}"] = metrics["per_class_f1"][label]

    row["checkpoint"] = path.relative_to(project_root).as_posix()
    summary_rows.append(row)

table_dir = project_root / "outputs" / "tables"
table_dir.mkdir(parents=True, exist_ok=True)
summary_path = table_dir / "brisc_seed42_validation_summary.csv"
temp_path = summary_path.with_suffix(".tmp")

with temp_path.open("w", newline="", encoding="utf-8") as file:
    writer = csv.DictWriter(file, fieldnames=list(summary_rows[0]))
    writer.writeheader()
    writer.writerows(summary_rows)

temp_path.replace(summary_path)
print("Saved:", summary_path)
print("Verified model results written:", len(summary_rows))

Saved: C:\Users\HP\Documents\Projects\Brain_Tumor_XAI\outputs\tables\brisc_seed42_validation_summary.csv
Verified model results written: 3


## Seed 123 — Second training run for each model

All three seed-42 checkpoints were verified on the 973-scan validation set. Their results are saved in `outputs/tables/brisc_seed42_validation_summary.csv`.

For seed 123, we train MobileNetV4 Conv Small, ConvNeXt V2 Atto, and RepViT M0.9 from their original pretrained weights. The new seed controls classifier initialization, training-image order, and augmentation draws. The data split, 2% training translation limit, and 15-epoch budget stay the same so we can compare results across seeds.

For each model, we select the epoch with the highest unrounded validation macro-F1. An exact tie goes to the earlier epoch.

In [33]:
def initialize_brisc_run(model_id, seed):
    pin = protocol["pretrained_checkpoints"][model_id]
    path = Path(hf_hub_download(
        repo_id=pin["repository"],
        revision=pin["revision"],
        filename=pin["filename"],
        local_files_only=True,
    ))
    if sha256(path.read_bytes()).hexdigest() != pin["sha256"]:
        raise ValueError(f"Pretrained checkpoint differs from protocol: {model_id}")

    with torch.random.fork_rng(devices=[]):
        torch.default_generator.manual_seed(seed)
        run_model = timm.create_model(
            model_id, pretrained=False, num_classes=num_classes
        )

    source = load_file(str(path), device="cpu")
    target = run_model.state_dict()
    assert source.keys() == target.keys(), "Checkpoint tensor names differ"

    fresh_names = []
    for name, tensor in source.items():
        if tensor.shape == target[name].shape:
            target[name] = tensor
            continue

        # Only classifier outputs may change from 1,000 classes to four.
        old_shape = tuple(tensor.shape)
        new_shape = tuple(target[name].shape)
        assert old_shape[:1] == (1000,) and new_shape == (num_classes,) + old_shape[1:], (
            f"Unexpected shape change for {name}: {old_shape} -> {new_shape}"
        )
        fresh_names.append(name)

    run_model.load_state_dict(target, strict=True)
    run_model = run_model.to(device)
    run_optimizer = torch.optim.AdamW(
        run_model.parameters(),
        lr=protocol["learning_rate"],
        weight_decay=protocol["weight_decay"],
        foreach=protocol["optimizer_options"]["foreach"],
    )
    return run_model, run_optimizer, fresh_names


seed123 = 123
mobile_id = "mobilenetv4_conv_small.e2400_r224_in1k"
mobile123_model, mobile123_optimizer, mobile123_fresh = initialize_brisc_run(
    mobile_id, seed123
)

assert set(mobile123_fresh) == {"classifier.weight", "classifier.bias"}
trainable_count = sum(p.numel() for p in mobile123_model.parameters() if p.requires_grad)
assert trainable_count == 2_498_148

mobile123_model.eval()
with torch.inference_mode():
    scores = mobile123_model(images.to(device))

assert scores.shape == (len(images), num_classes)
assert torch.isfinite(scores).all().item()

print("Model:", mobile_id)
print("Run seed:", seed123)
print("Trainable parameters:", f"{trainable_count:,}")
print("GPU output shape:", tuple(scores.shape))
print("PASS | MobileNetV4 initialized for seed 123")

Model: mobilenetv4_conv_small.e2400_r224_in1k
Run seed: 123
Trainable parameters: 2,498,148
GPU output shape: (8, 4)
PASS | MobileNetV4 initialized for seed 123


In [34]:
def run_training_schedule(active_model, active_optimizer, model_id, seed, run_dir):
    run_dir.mkdir(parents=True, exist_ok=True)
    final_epoch = protocol["epochs_per_run"]
    shuffle = torch.Generator().manual_seed(seed)

    def save_epoch(number, history):
        path = run_dir / f"epoch_{number:02d}.pt"
        temp = path.with_suffix(".tmp")
        try:
            torch.save({
                "model_id": model_id,
                "epoch": number,
                "run_seed": seed,
                "class_to_index": class_to_index,
                "model_state_dict": active_model.state_dict(),
                "optimizer_state_dict": active_optimizer.state_dict(),
                "shuffle_generator_state": shuffle.get_state(),
                "torch_cpu_rng_state": torch.get_rng_state(),
                "torch_cuda_rng_state": torch.cuda.get_rng_state(device),
                "protocol": protocol,
                "training_augmentation": aug_policy,
                "metrics": history.get(number, {}),
                "history": history,
            }, temp)
            temp.replace(path)
        finally:
            temp.unlink(missing_ok=True)

    checkpoints = sorted(run_dir.glob("epoch_*.pt"))
    if checkpoints:
        latest_path = checkpoints[-1]
        state = torch.load(latest_path, map_location="cpu", weights_only=True)
        assert state["model_id"] == model_id and state["run_seed"] == seed
        assert state["protocol"] == protocol
        assert state["training_augmentation"] == aug_policy
        assert state["class_to_index"] == class_to_index

        active_model.load_state_dict(state["model_state_dict"], strict=True)
        active_optimizer.load_state_dict(state["optimizer_state_dict"])
        shuffle.set_state(state["shuffle_generator_state"])
        torch.set_rng_state(state["torch_cpu_rng_state"])
        torch.cuda.set_rng_state(state["torch_cuda_rng_state"], device=device)
        completed = state["epoch"]
        history = state["history"]
        del state
    else:
        torch.manual_seed(seed)
        completed = 0
        history = {}
        save_epoch(0, history)

    print(f"Seed {seed} | {completed}/{final_epoch} epochs completed", flush=True)

    for number in range(completed + 1, final_epoch + 1):
        batches = make_training_batches(n_train, batch_size, shuffle)
        train_loss, train_seconds = train_model_epoch(
            active_model, active_optimizer, seed, number, batches
        )
        metrics = validate_model(active_model)
        metrics["train_loss"] = train_loss
        metrics["train_seconds"] = train_seconds
        history[number] = metrics
        save_epoch(number, history)

        print(
            f"Epoch {number:02d}/{final_epoch} | train loss {train_loss:.4f} | "
            f"val loss {metrics['validation_loss']:.4f} | "
            f"val acc {metrics['validation_accuracy']:.2%} | "
            f"macro-F1 {metrics['validation_macro_f1']:.6f} | saved",
            flush=True,
        )

    best_epoch = max(sorted(history), key=lambda n: history[n]["validation_macro_f1"])
    best_path = run_dir / f"epoch_{best_epoch:02d}.pt"
    print(f"\nBest epoch: {best_epoch}")
    print("Best validation macro-F1:", history[best_epoch]["validation_macro_f1"])
    print("Selected checkpoint:", best_path)
    return history, best_path


mobile123_dir = checkpoint_dir / "mobilenetv4_conv_small_seed123"
mobile123_history, mobile123_best_path = run_training_schedule(
    mobile123_model, mobile123_optimizer, mobile_id, seed123, mobile123_dir
)

Seed 123 | 0/15 epochs completed
Completed 100/487 batches
Completed 200/487 batches
Completed 300/487 batches
Completed 400/487 batches
Epoch 01/15 | train loss 1.4458 | val loss 0.4747 | val acc 84.17% | macro-F1 0.837772 | saved
Completed 100/487 batches
Completed 200/487 batches
Completed 300/487 batches
Completed 400/487 batches
Epoch 02/15 | train loss 0.5707 | val loss 0.5035 | val acc 85.51% | macro-F1 0.849312 | saved
Completed 100/487 batches
Completed 200/487 batches
Completed 300/487 batches
Completed 400/487 batches
Epoch 03/15 | train loss 0.4315 | val loss 0.4032 | val acc 87.98% | macro-F1 0.874913 | saved
Completed 100/487 batches
Completed 200/487 batches
Completed 300/487 batches
Completed 400/487 batches
Epoch 04/15 | train loss 0.3474 | val loss 0.4936 | val acc 88.59% | macro-F1 0.880064 | saved
Completed 100/487 batches
Completed 200/487 batches
Completed 300/487 batches
Completed 400/487 batches
Epoch 05/15 | train loss 0.3300 | val loss 0.3812 | val acc 88.28% 

In [35]:
mobile123_selected = torch.load(
    mobile123_best_path, map_location="cpu", weights_only=True
)

assert mobile123_selected["model_id"] == mobile_id
assert mobile123_selected["run_seed"] == seed123
assert mobile123_selected["class_to_index"] == class_to_index

expected_epoch = max(
    sorted(mobile123_history),
    key=lambda n: mobile123_history[n]["validation_macro_f1"],
)
assert mobile123_selected["epoch"] == expected_epoch
mobile123_stored = mobile123_selected["metrics"]
assert mobile123_stored == mobile123_history[expected_epoch]

mobile123_model.load_state_dict(mobile123_selected["model_state_dict"], strict=True)
mobile123_verified = validate_model(mobile123_model)

print("Selected checkpoint:", mobile123_best_path.name)
print("Stored macro-F1:    ", mobile123_stored["validation_macro_f1"])
print("Recomputed macro-F1:", mobile123_verified["validation_macro_f1"])

for metric in ("validation_macro_f1", "validation_accuracy", "per_class_f1"):
    assert mobile123_verified[metric] == mobile123_stored[metric], (
        f"Recomputed {metric} differs from the checkpoint"
    )

print(f"Validation accuracy: {mobile123_verified['validation_accuracy']:.2%}")
for label, score in mobile123_verified["per_class_f1"].items():
    print(f"  {label} F1: {score:.4f}")
print("PASS | MobileNetV4 seed-123 checkpoint reproduces its validation scores")

Selected checkpoint: epoch_13.pt
Stored macro-F1:     0.946728118652276
Recomputed macro-F1: 0.946728118652276
Validation accuracy: 94.66%
  glioma F1: 0.9424
  meningioma F1: 0.9087
  no_tumor F1: 0.9809
  pituitary F1: 0.9549
PASS | MobileNetV4 seed-123 checkpoint reproduces its validation scores


In [36]:
convnext123_model, convnext123_optimizer, convnext123_fresh = initialize_brisc_run(
    convnext_id, seed123
)

assert set(convnext123_fresh) == {"head.fc.weight", "head.fc.bias"}
trainable_count = sum(
    p.numel() for p in convnext123_model.parameters() if p.requires_grad
)
assert trainable_count == 3_388_684

convnext123_model.eval()
with torch.inference_mode():
    preview_scores = convnext123_model(images.to(device))

assert preview_scores.shape == (len(images), num_classes)
assert torch.isfinite(preview_scores).all().item()

convnext123_dir = checkpoint_dir / "convnextv2_atto_seed123"
print(f"ConvNeXt V2 Atto | seed {seed123} | {trainable_count:,} trainable parameters")
print("Checkpoint folder:", convnext123_dir, flush=True)

convnext123_history, convnext123_best_path = run_training_schedule(
    convnext123_model,
    convnext123_optimizer,
    convnext_id,
    seed123,
    convnext123_dir,
)

ConvNeXt V2 Atto | seed 123 | 3,388,684 trainable parameters
Checkpoint folder: C:\Users\HP\Documents\Projects\Brain_Tumor_XAI\outputs\checkpoints\convnextv2_atto_seed123
Seed 123 | 0/15 epochs completed
Completed 100/487 batches
Completed 200/487 batches
Completed 300/487 batches
Completed 400/487 batches
Epoch 01/15 | train loss 0.3659 | val loss 0.1257 | val acc 96.30% | macro-F1 0.964326 | saved
Completed 100/487 batches
Completed 200/487 batches
Completed 300/487 batches
Completed 400/487 batches
Epoch 02/15 | train loss 0.1323 | val loss 0.1166 | val acc 96.09% | macro-F1 0.961078 | saved
Completed 100/487 batches
Completed 200/487 batches
Completed 300/487 batches
Completed 400/487 batches
Epoch 03/15 | train loss 0.1057 | val loss 0.0919 | val acc 97.43% | macro-F1 0.974184 | saved
Completed 100/487 batches
Completed 200/487 batches
Completed 300/487 batches
Completed 400/487 batches
Epoch 04/15 | train loss 0.0720 | val loss 0.1048 | val acc 95.68% | macro-F1 0.956714 | saved


In [37]:
convnext123_selected = torch.load(
    convnext123_best_path, map_location="cpu", weights_only=True
)

expected_epoch = max(
    sorted(convnext123_history),
    key=lambda n: convnext123_history[n]["validation_macro_f1"],
)
assert convnext123_selected["model_id"] == convnext_id
assert convnext123_selected["run_seed"] == seed123
assert convnext123_selected["epoch"] == expected_epoch
assert convnext123_selected["class_to_index"] == class_to_index

convnext123_stored = convnext123_selected["metrics"]
assert convnext123_stored == convnext123_history[expected_epoch]

convnext123_model.load_state_dict(
    convnext123_selected["model_state_dict"], strict=True
)
convnext123_verified = validate_model(convnext123_model)

print("Selected checkpoint:", convnext123_best_path.name)
print("Stored macro-F1:    ", convnext123_stored["validation_macro_f1"])
print("Recomputed macro-F1:", convnext123_verified["validation_macro_f1"])

for metric in ("validation_macro_f1", "validation_accuracy", "per_class_f1"):
    assert convnext123_verified[metric] == convnext123_stored[metric], (
        f"Recomputed {metric} differs from the checkpoint"
    )

print(f"Validation accuracy: {convnext123_verified['validation_accuracy']:.2%}")
for label, score in convnext123_verified["per_class_f1"].items():
    print(f"  {label} F1: {score:.4f}")
print("PASS | ConvNeXt seed-123 checkpoint reproduces its validation scores")

Selected checkpoint: epoch_08.pt
Stored macro-F1:     0.9845523595076718
Recomputed macro-F1: 0.9845523595076718
Validation accuracy: 98.46%
  glioma F1: 0.9762
  meningioma F1: 0.9713
  no_tumor F1: 0.9976
  pituitary F1: 0.9930
PASS | ConvNeXt seed-123 checkpoint reproduces its validation scores


In [38]:
repvit123_model, repvit123_optimizer, repvit123_fresh = initialize_brisc_run(
    repvit_id, seed123
)

expected_heads = {
    "head.head.l.bias",
    "head.head.l.weight",
    "head.head_dist.l.bias",
    "head.head_dist.l.weight",
}
assert set(repvit123_fresh) == expected_heads, "Unexpected RepViT classifier tensors"

trainable_count = sum(p.numel() for p in repvit123_model.parameters() if p.requires_grad)
assert trainable_count == 4_722_408

repvit123_model.eval()
with torch.inference_mode():
    preview_scores = repvit123_model(images.to(device))

assert preview_scores.shape == (len(images), num_classes)
assert torch.isfinite(preview_scores).all().item()

repvit123_dir = checkpoint_dir / "repvit_m0_9_seed123"
print(f"RepViT M0.9 | seed {seed123} | {trainable_count:,} trainable parameters")
print("Checkpoint folder:", repvit123_dir, flush=True)

repvit123_history, repvit123_best_path = run_training_schedule(
    repvit123_model,
    repvit123_optimizer,
    repvit_id,
    seed123,
    repvit123_dir,
)

RepViT M0.9 | seed 123 | 4,722,408 trainable parameters
Checkpoint folder: C:\Users\HP\Documents\Projects\Brain_Tumor_XAI\outputs\checkpoints\repvit_m0_9_seed123
Seed 123 | 0/15 epochs completed
Completed 100/487 batches
Completed 200/487 batches
Completed 300/487 batches
Completed 400/487 batches
Epoch 01/15 | train loss 0.5008 | val loss 0.2525 | val acc 91.68% | macro-F1 0.914177 | saved
Completed 100/487 batches
Completed 200/487 batches
Completed 300/487 batches
Completed 400/487 batches
Epoch 02/15 | train loss 0.2012 | val loss 0.1254 | val acc 96.20% | macro-F1 0.961773 | saved
Completed 100/487 batches
Completed 200/487 batches
Completed 300/487 batches
Completed 400/487 batches
Epoch 03/15 | train loss 0.1432 | val loss 0.0821 | val acc 97.94% | macro-F1 0.979925 | saved
Completed 100/487 batches
Completed 200/487 batches
Completed 300/487 batches
Completed 400/487 batches
Epoch 04/15 | train loss 0.0916 | val loss 0.0696 | val acc 97.53% | macro-F1 0.975896 | saved
Completed

In [2]:
import csv
from hashlib import sha256
from pathlib import Path

import timm
import torch
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms as T
from torchvision.transforms.functional import pil_to_tensor

project_root = Path(r"C:\Users\HP\Documents\Projects\Brain_Tumor_XAI")
checkpoint_dir = project_root / "outputs" / "checkpoints"
repvit123_best_path = checkpoint_dir / "repvit_m0_9_seed123" / "epoch_13.pt"

assert torch.cuda.is_available(), "CUDA unavailable. Check the selected kernel."
device = torch.device("cuda:0")
repvit123_selected = torch.load(
    repvit123_best_path, map_location="cpu", weights_only=True
)

repvit_id = "repvit_m0_9.dist_450e_in1k"
assert repvit123_selected["model_id"] == repvit_id
assert repvit123_selected["run_seed"] == 123
assert repvit123_selected["epoch"] == 13

protocol = repvit123_selected["protocol"]
class_to_index = repvit123_selected["class_to_index"]
repvit123_stored = repvit123_selected["metrics"]

# Verify the exact validation CSV used during training.
manifest = project_root / "data" / "processed" / "classification_validation_manifest.csv"
assert sha256(manifest.read_bytes()).hexdigest() == (
    protocol["development_manifest_sha256"]["validation"]
), "Validation manifest differs from the training run"

with manifest.open(encoding="utf-8-sig", newline="") as file:
    validation_rows = list(csv.DictReader(file))
assert len(validation_rows) == protocol["validation"]["rows_per_epoch"]

dataset_root = project_root / "data" / "raw" / "brisc2025"
normalize = T.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))

class RecoveryValidationDataset(Dataset):
    def __len__(self):
        return len(validation_rows)

    def __getitem__(self, index):
        row = validation_rows[index]
        with Image.open(dataset_root / row["relative_path"]) as image:
            gray = image.convert("L").resize((224, 224), Image.Resampling.BILINEAR)
        pixels = pil_to_tensor(gray).to(torch.float32) / 255.0
        return normalize(pixels.repeat(3, 1, 1)), class_to_index[row["tumor_label"]]

loader = DataLoader(
    RecoveryValidationDataset(),
    batch_size=protocol["batch_size"],
    shuffle=False,
    num_workers=0,
)

repvit123_model = timm.create_model(repvit_id, pretrained=False, num_classes=4)
repvit123_model.load_state_dict(repvit123_selected["model_state_dict"], strict=True)
repvit123_model.to(device).eval()

targets, predictions = [], []
with torch.inference_mode():
    for images, labels in loader:
        scores = repvit123_model(images.to(device))
        assert torch.isfinite(scores).all().item(), "Non-finite model output"
        targets.extend(labels.tolist())
        predictions.extend(scores.argmax(dim=1).cpu().tolist())

assert len(predictions) == len(validation_rows)
y_true = torch.tensor(targets)
y_pred = torch.tensor(predictions)
accuracy = (y_true == y_pred).sum().item() / len(y_true)

per_class = {}
for name, index in class_to_index.items():
    actual = y_true == index
    predicted = y_pred == index
    true_positives = (actual & predicted).sum().item()
    denominator = actual.sum().item() + predicted.sum().item()
    per_class[name] = 2 * true_positives / denominator if denominator else 0.0

macro_f1 = sum(per_class.values()) / len(per_class)
print("Checkpoint:", repvit123_best_path.name)
print("Stored macro-F1:    ", repvit123_stored["validation_macro_f1"])
print("Recomputed macro-F1:", macro_f1)
print(f"Validation accuracy: {accuracy:.2%} across {len(y_true)} scans")
for name, score in per_class.items():
    print(f"  {name} F1: {score:.4f}")

assert macro_f1 == repvit123_stored["validation_macro_f1"]
assert accuracy == repvit123_stored["validation_accuracy"]
assert per_class == repvit123_stored["per_class_f1"]
print("PASS | RepViT seed-123 checkpoint verified after kernel restart")

C:\Users\HP\miniconda3\envs\brainxai-train\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Checkpoint: epoch_13.pt
Stored macro-F1:     0.9919189347234519
Recomputed macro-F1: 0.9919189347234519
Validation accuracy: 99.18% across 973 scans
  glioma F1: 0.9912
  meningioma F1: 0.9857
  no_tumor F1: 0.9976
  pituitary F1: 0.9931
PASS | RepViT seed-123 checkpoint verified after kernel restart


In [3]:
import csv

seed123_checkpoints = [
    ("RepViT M0.9", "repvit_m0_9.dist_450e_in1k", "repvit_m0_9_seed123", 13),
    ("ConvNeXt V2 Atto", "convnextv2_atto.fcmae_ft_in1k", "convnextv2_atto_seed123", 8),
    ("MobileNetV4 Conv Small", "mobilenetv4_conv_small.e2400_r224_in1k",
     "mobilenetv4_conv_small_seed123", 13),
]

summary_rows = []
for name, model_id, folder, epoch in seed123_checkpoints:
    path = checkpoint_dir / folder / f"epoch_{epoch:02d}.pt"
    state = torch.load(path, map_location="cpu", weights_only=True)
    assert (state["model_id"], state["run_seed"], state["epoch"]) == (model_id, 123, epoch)
    assert state["class_to_index"] == class_to_index

    metrics = state["metrics"]
    row = {
        "model": name,
        "model_id": model_id,
        "seed": 123,
        "selected_epoch": epoch,
        "validation_scans": len(validation_rows),
        "validation_accuracy": metrics["validation_accuracy"],
        "validation_macro_f1": metrics["validation_macro_f1"],
        "validation_loss": metrics["validation_loss"],
    }
    for label in class_to_index:
        row[f"f1_{label}"] = metrics["per_class_f1"][label]

    row["checkpoint"] = path.relative_to(project_root).as_posix()
    summary_rows.append(row)
    del state

table_dir = project_root / "outputs" / "tables"
table_dir.mkdir(parents=True, exist_ok=True)
summary_path = table_dir / "brisc_seed123_validation_summary.csv"
temp_path = summary_path.with_suffix(".tmp")

with temp_path.open("w", encoding="utf-8", newline="") as file:
    writer = csv.DictWriter(file, fieldnames=list(summary_rows[0]))
    writer.writeheader()
    writer.writerows(summary_rows)

temp_path.replace(summary_path)
print("Saved:", summary_path)
print("Verified model results written:", len(summary_rows))

Saved: C:\Users\HP\Documents\Projects\Brain_Tumor_XAI\outputs\tables\brisc_seed123_validation_summary.csv
Verified model results written: 3


## Seed 2026 — Final training runs for all three models

The seed-42 and seed-123 runs are complete. All six selected checkpoints reproduced their validation scores after reloading. The results are saved in `brisc_seed42_validation_summary.csv` and `brisc_seed123_validation_summary.csv`.

For the final seed, we train MobileNetV4 Conv Small, ConvNeXt V2 Atto, and RepViT M0.9 for 15 epochs each. We use the same training and validation splits and the same 2% translation limit for training. For each model, we select the earliest epoch with the highest unrounded validation macro-F1 and verify the saved checkpoint on all 973 validation scans.

In [4]:
import ast
import csv
import json
from time import perf_counter

import torch.nn.functional as F
from huggingface_hub import hf_hub_download
from safetensors.torch import load_file
from torchvision.transforms import InterpolationMode

# Recover the definitions already used for the verified runs.
notebook_path = (
    project_root / "notebooks" / "09_BRISC_Training_Setup_and_Timed_Epoch.ipynb"
)
notebook = json.loads(notebook_path.read_text(encoding="utf-8"))
needed = {
    "load_canonical_mri", "augment_training_image", "make_training_batches",
    "TrainingEpochDataset", "ValidationDataset", "train_model_epoch",
    "validate_model", "initialize_brisc_run", "run_training_schedule",
}
definitions = {}
for cell in notebook["cells"]:
    if cell["cell_type"] != "code":
        continue
    source = "".join(cell["source"])
    for node in ast.parse(source).body:
        if isinstance(node, (ast.FunctionDef, ast.ClassDef)) and node.name in needed:
            definitions[node.name] = node

missing = needed - definitions.keys()
assert not missing, f"Definitions missing from saved notebook: {sorted(missing)}"
module = ast.Module(body=list(definitions.values()), type_ignores=[])
exec(compile(module, str(notebook_path), "exec"), globals())

# Restore the same data and preprocessing that the training functions expect.
processed_dir = project_root / "data" / "processed"
records = {"validation": validation_rows}
train_path = processed_dir / "classification_train_manifest.csv"
assert sha256(train_path.read_bytes()).hexdigest() == (
    protocol["development_manifest_sha256"]["train"]
)
with train_path.open(encoding="utf-8-sig", newline="") as file:
    records["train"] = list(csv.DictReader(file))

config_bytes = (processed_dir / "brisc_preprocessing_config.json").read_bytes()
assert sha256(config_bytes).hexdigest() == protocol["augmentation"]["base_config_sha256"]
image_size = json.loads(config_bytes)["classification"]["default_input_size"]
aug_policy = repvit123_selected["training_augmentation"]
translation = aug_policy["translation_fraction"]
assert (translation["horizontal"], translation["vertical"]) == (0.02, 0.02)

train_augmentation = T.Compose([
    T.RandomAffine(
        degrees=tuple(aug_policy["rotation_degrees"]),
        translate=(translation["horizontal"], translation["vertical"]),
        scale=tuple(aug_policy["scale_range"]),
        interpolation=InterpolationMode.BILINEAR,
        fill=0,
    ),
    T.ColorJitter(
        brightness=tuple(aug_policy["brightness_factor_range"]),
        contrast=tuple(aug_policy["contrast_factor_range"]),
    ),
])

checkpoint_normalize = normalize
batch_size = protocol["batch_size"]
num_classes = len(class_to_index)
n_train = len(records["train"])
criterion = torch.nn.CrossEntropyLoss()
seed2026 = 2026
validation_loader = DataLoader(
    ValidationDataset(validation_rows),
    batch_size=batch_size,
    shuffle=False,
    num_workers=0,
    generator=torch.Generator().manual_seed(seed2026),
)

runs = [
    ("MobileNetV4 Conv Small", "mobilenetv4_conv_small.e2400_r224_in1k",
     "mobilenetv4_conv_small_seed2026", 2_498_148),
    ("ConvNeXt V2 Atto", "convnextv2_atto.fcmae_ft_in1k",
     "convnextv2_atto_seed2026", 3_388_684),
    ("RepViT M0.9", "repvit_m0_9.dist_450e_in1k",
     "repvit_m0_9_seed2026", 4_722_408),
]

summary_2026 = []
for name, model_id, folder, expected_count in runs:
    print(f"\nStarting {name} | seed {seed2026}", flush=True)
    active_model, active_optimizer, _ = initialize_brisc_run(model_id, seed2026)
    count = sum(p.numel() for p in active_model.parameters() if p.requires_grad)
    assert count == expected_count, f"Unexpected parameter count for {name}"

    history, best_path = run_training_schedule(
        active_model, active_optimizer, model_id, seed2026, checkpoint_dir / folder
    )

    # Verify the selected weights on all 973 validation scans.
    best = torch.load(best_path, map_location="cpu", weights_only=True)
    assert best["model_id"] == model_id and best["run_seed"] == seed2026
    active_model.load_state_dict(best["model_state_dict"], strict=True)
    checked = validate_model(active_model)
    metrics = best["metrics"]
    for key in ("validation_macro_f1", "validation_accuracy", "per_class_f1"):
        assert checked[key] == metrics[key], f"{name}: verification failed for {key}"

    row = {
        "model": name,
        "model_id": model_id,
        "seed": seed2026,
        "selected_epoch": best["epoch"],
        "validation_scans": len(validation_rows),
        "validation_accuracy": metrics["validation_accuracy"],
        "validation_macro_f1": metrics["validation_macro_f1"],
        "validation_loss": metrics["validation_loss"],
    }
    for label in class_to_index:
        row[f"f1_{label}"] = metrics["per_class_f1"][label]
    row["checkpoint"] = best_path.relative_to(project_root).as_posix()
    summary_2026.append(row)

    print(f"VERIFIED | {name} | epoch {best['epoch']} | "
          f"macro-F1 {metrics['validation_macro_f1']:.6f}", flush=True)
    del active_model, active_optimizer, best

summary_path = table_dir / "brisc_seed2026_validation_summary.csv"
temp_path = summary_path.with_suffix(".tmp")
with temp_path.open("w", encoding="utf-8", newline="") as file:
    writer = csv.DictWriter(file, fieldnames=list(summary_2026[0]))
    writer.writeheader()
    writer.writerows(summary_2026)
temp_path.replace(summary_path)

print("\nSaved:", summary_path)
print("Verified seed-2026 results written:", len(summary_2026))


Starting MobileNetV4 Conv Small | seed 2026
Seed 2026 | 0/15 epochs completed
Completed 100/487 batches
Completed 200/487 batches
Completed 300/487 batches
Completed 400/487 batches
Epoch 01/15 | train loss 1.6421 | val loss 0.9753 | val acc 80.47% | macro-F1 0.790937 | saved
Completed 100/487 batches
Completed 200/487 batches
Completed 300/487 batches
Completed 400/487 batches
Epoch 02/15 | train loss 0.6702 | val loss 0.4474 | val acc 88.28% | macro-F1 0.880759 | saved
Completed 100/487 batches
Completed 200/487 batches
Completed 300/487 batches
Completed 400/487 batches
Epoch 03/15 | train loss 0.5023 | val loss 0.2989 | val acc 88.90% | macro-F1 0.887383 | saved
Completed 100/487 batches
Completed 200/487 batches
Completed 300/487 batches
Completed 400/487 batches
Epoch 04/15 | train loss 0.4156 | val loss 0.3386 | val acc 90.03% | macro-F1 0.897438 | saved
Completed 100/487 batches
Completed 200/487 batches
Completed 300/487 batches
Completed 400/487 batches
Epoch 05/15 | train l

## Independent review of the training translation limit

Shruti Singh inspected the saved augmentation comparison in Notebook 08. She compared the original images with paired augmentation draws using maximum translations of 5% and 2% per axis (seed 42).

- **Pituitary scan 04573:** The original right margin was already narrow. The 5% setting tightened it further. The 2% setting preserved more of the margin, although the edge remained tight. Shruti observed no clear substantial tissue clipping.
- **No-tumor scan 02790:** The 5% setting narrowed the space above the skull. The 2% setting preserved more of that space, and the upper skull boundary remained visible.

All nine classification runs used a 2% translation limit per axis for training. Shruti’s subsequent review supported retaining that setting. The evidence covers two selected scans and one paired random draw. A broader assessment would require more scans and draws. Changes to cropping or resizing would warrant a fresh comparison.

## Training conclusion — Nine runs completed

We trained three pretrained CNNs on BRISC using seeds 42, 123, and 2026. Each model completed 15 epochs per seed, for a total of 9 runs and 135 training epochs. All runs used the same split of 3,889 training scans and 973 validation scans, with a maximum training translation of 2% per axis.

For each run, we selected the checkpoint with the highest unrounded validation macro-F1, choosing the earlier epoch in the event of a tie. We reloaded all nine selected checkpoints and reproduced their recorded validation scores.

| Model | Seed 42 macro-F1 (epoch) | Seed 123 macro-F1 (epoch) | Seed 2026 macro-F1 (epoch) |
|---|---:|---:|---:|
| MobileNetV4 Conv Small | 0.954777 (14) | 0.946728 (13) | 0.935038 (5) |
| ConvNeXt V2 Atto | 0.986645 (10) | 0.984552 (8) | 0.984696 (14) |
| RepViT M0.9 | 0.992997 (15) | 0.991919 (13) | 0.991962 (8) |

RepViT M0.9 achieved the highest validation macro-F1 for all three seeds, followed by ConvNeXt V2 Atto and MobileNetV4 Conv Small. The selected checkpoints are saved under `outputs/checkpoints`, and the three seed-specific validation summaries are saved under `outputs/tables`.

Training is complete. The next stage covers learning-curve analysis, evaluation on the held-out test set, and comparison of the models’ XAI overlays.